# V3 / 02B — Overnight model strategy benchmark

**Primary objective:** optimal, calibrated Musanga probability maps of all existing UAV orthomosaics. Three within-forest spatial folds benchmark four pooled strategies; then compare site-specific fine-tuning and all-site leave-one-site-out transfer. MOS 1–2/3–4/5–6 form a single forest group. Outputs checkpointed to Drive; original inputs unchanged.


## 01 — GPU, existing split tables, user settings


In [1]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OUT=ROOT/'Outputs/V3/02_overnight';OUT.mkdir(parents=True,exist_ok=True)
REF=ROOT/'Outputs/V3/01_reference_data'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
mosaics={m['file']:m for m in A['mosaics']}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
FOLD=0;SEED=20261015;CROP=256;EPOCHS=16;BATCH=8;MAX_PER_SITE_KIND=90
FOLDS=[0,1,2]
STRATEGIES=['exhaustive','combined','combined_light','combined_balanced']
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)




Mounted at /content/drive
GPU Tesla T4 | fold 0
Train/val candidates: {'train': 2538, 'val': 285}
Files: /content/drive/MyDrive/Colab Notebooks/Musanga/Final | output: /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3/02_overnight


## 02 — Authoritative reference sources and masks; MOS is one forest group


In [2]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>.05 or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))
def make_masks(name,window,role,include_partial):
    d=geo[name];bounds=box(*rasterio.windows.bounds(window,d['transform']))
    affine=rasterio.windows.transform(window,d['transform'])
    size=(int(window.height),int(window.width))
    def burn(series):
        gs=[g.intersection(bounds) for g in series if g is not None and g.intersects(bounds)]
        return rasterize([(g,1) for g in gs if not g.is_empty],out_shape=size,
            transform=affine,fill=0,dtype='uint8').astype(bool) if gs else np.zeros(size,bool)
    model_roi=burn(d['roi'].geometry)
    cs=d['circles']
    selected=cs[(cs.fold!=FOLD) if role=='train' else (cs.fold==FOLD)]
    # Only the selected mosaic owns the exhaustive circle.
    selected=selected[selected.owner==name]
    full=burn(selected.geometry)&model_roi
    pos=burn(d['positive'].geometry)&full
    partpos=burn(d['trees'].loc[d['trees'].class01==1].geometry)
    partneg=burn(d['trees'].loc[d['trees'].class01==0].geometry)
    contradictions=(pos&partneg)|(full&~pos&partpos)|(~full&partpos&partneg)
    y=(pos|(~full&partpos)).astype('float32')
    if role=='val':
        weight=full
    else:
        weight=full | ((partpos|partneg)&~full if include_partial else np.zeros(size,bool))
        weight &= ~contradictions
        # All heldout circles in this forest group, even if assigned to another MOS mosaic
        heldout=cs[cs.fold==FOLD]
        blocked=burn([g.buffer(15.) for g in heldout.geometry])
        weight &= ~blocked
    weight &= model_roi
    return y,weight.astype('float32'),full,contradictions
print('Window mask generator ready. Original vector sources remain untouched.')



Window mask generator ready. Original vector sources remain untouched.


## 03 — Small U-Net and masked binary cross-entropy


In [3]:
class TinyUNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
        self.e1=block(3,base);self.e2=block(base,base*2)
        self.bridge=block(base*2,base*4)
        self.d2=block(base*4+base*2,base*2)
        self.d1=block(base*2+base,base)
        self.final=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(F.max_pool2d(e1,2))
        b=self.bridge(F.max_pool2d(e2,2))
        x=self.d2(torch.cat([F.interpolate(b,size=e2.shape[-2:],mode='bilinear',align_corners=False),e2],1))
        x=self.d1(torch.cat([F.interpolate(x,size=e1.shape[-2:],mode='bilinear',align_corners=False),e1],1))
        return self.final(x)[:,0]
class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Tiny U-Net ready (native 256×256 RGB crops).')



Tiny U-Net ready (native 256×256 RGB crops).


## 04 — Cached training chips by fold

The training RGB chip is excluded if it intersects **any** held-out circle (+15 m buffer) in its forest group, including circles assigned to another MOS mosaic. This prevents cross-mosaic visual leakage.


In [4]:
def choose_windows(split,kind=None,cap_per_site=35):
    a=splits[(splits.cv_fold==FOLD)&(splits.split==split)].copy()
    if kind is not None:a=a[a.kind==kind]
    result=[]
    for site,g in a.groupby('site'):
        # Include diverse mosaics; reproducible within-site shuffle.
        g=g.sample(frac=1,random_state=SEED)
        result.extend(g.iloc[:min(cap_per_site,len(g))].to_dict('records'))
    return result
def load_crop(rec,role,combined=True):
    name=rec['mosaic']
    row=int(rec['row'])+max(0,(int(rec['height'])-CROP)//2)
    col=int(rec['col'])+max(0,(int(rec['width'])-CROP)//2)
    if int(rec['height'])<CROP or int(rec['width'])<CROP:return None
    window=Window(col,row,CROP,CROP)
    y,w,full,unc=make_masks(name,window,role,include_partial=combined)
    if role=='train':
        site=geo[name]['site']
        cg=circles.to_crs(geo[name]['crs'])
        held=set(links[(links.site==site)&(links.fold==FOLD)].circle_id.astype(int))
        tile=box(*rasterio.windows.bounds(window,geo[name]['transform']))
        if any(tile.intersects(g.buffer(15.)) for g in cg[cg.original_row.isin(held)].geometry):return None
    if role=='train' and float(w.sum())<500:return None
    if role=='val' and float(w.sum())<200:return None
    with rasterio.open(geo[name]['path']) as ds:
        rgb=ds.read([1,2,3],window=window,masked=True)
    band_ok=~np.any(np.ma.getmaskarray(rgb),axis=0)
    raw=rgb.filled(0).astype('float32')
    rgb_ok=band_ok&np.any(raw!=0,axis=0)&np.any(raw!=255,axis=0)
    # Inference must use the original data validity; unknown pixels never train.
    w *= rgb_ok
    if float(w.sum())<200:return None
    rgb=np.clip(raw/255.,0,1).transpose(1,2,0)
    return dict(rgb=rgb.astype('float32'),y=y,w=w.astype('float32'),
                full=full,unc=unc,site=geo[name]['site'],name=name,
                circle_id=int(rec['circle_id']),kind=rec['kind'])

def prepare_fold(fold):
    global FOLD,train_cache
    FOLD=fold
    training=choose_windows('train',None,MAX_PER_SITE_KIND*2)
    train_cache=[v for x in training if (v:=load_crop(x,'train',combined=True)) is not None]
    assert train_cache,'No usable training crops'
    val=splits[(splits.cv_fold==fold)&(splits.split=='val')&
               (splits.kind=='exhaustive')]
    assert val.site.nunique()==4,'Validation circles must cover all four forests'
    print('Fold',fold,'train chips',len(train_cache),'complete-circle validation windows',len(val),flush=True)
    print('Train per site:',pd.Series([x['site'] for x in train_cache]).value_counts().to_dict(),flush=True)
    return train_cache,val



## 05 — Model strategies, illumination and balanced sampling

All strategies share the same architecture, optimiser, epochs and fold. `combined_light` uses moderate RGB brightness/contrast/gamma augmentation; `combined_balanced` resamples site × annotation type combinations. Values outside trusted masks never contribute to the loss.


In [5]:
from torch.utils.data import WeightedRandomSampler
class StrategyData(Dataset):
    def __init__(self,rows,strategy):
        self.rows=rows;self.strategy=strategy
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];x=d['rgb'].copy();y=d['y'].copy();w=d['w'].copy()
        if self.strategy=='exhaustive':w *= d['full']*(~d['unc'])
        k=int(torch.randint(0,4,(1,)).item())
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if torch.rand(1).item()<.5:
            x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        if self.strategy=='combined_light':
            b=float(torch.empty(1).uniform_(.8,1.2))
            c=float(torch.empty(1).uniform_(.85,1.15))
            gamma=float(torch.empty(1).uniform_(.8,1.2))
            x=np.clip((x-.5)*c+.5,0,1)
            x=np.clip(x*b,0,1)**gamma
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))).float(),
                torch.from_numpy(np.ascontiguousarray(y)).float(),
                torch.from_numpy(np.ascontiguousarray(w)).float())
def train_strategy(strategy,fold,cache):
    rows=cache
    if strategy=='exhaustive':
        rows=[d for d in cache if (d['w']*d['full']*(~d['unc'])).sum()>200]
    assert len(rows)>=8,(strategy,fold,len(rows))
    dataset=StrategyData(rows,strategy)
    if strategy=='combined_balanced':
        cats=[(d['site'],d['kind']) for d in rows]
        freq=pd.Series(cats).value_counts().to_dict()
        weights=torch.tensor([1/freq[k] for k in cats],dtype=torch.double)
        sampler=WeightedRandomSampler(weights,num_samples=len(rows),replacement=True)
        loader=DataLoader(dataset,batch_size=BATCH,sampler=sampler,num_workers=0)
    else:loader=DataLoader(dataset,batch_size=BATCH,shuffle=True,num_workers=0)
    torch.manual_seed(SEED)
    net=TinyUNet().to(device)
    optimizer=torch.optim.AdamW(net.parameters(),lr=7e-4,weight_decay=1e-4)
    logs=[]
    for epoch in range(EPOCHS):
        net.train();losses=[];start=time.time()
        for x,y,w in loader:
            x=x.to(device);y=y.to(device);w=w.to(device)
            if w.sum().item()==0:continue
            optimizer.zero_grad(set_to_none=True)
            z=net(x);loss=masked_loss(z,y,w)
            loss.backward();optimizer.step()
            losses.append(float(loss.item()))
        loss_epoch=float(np.mean(losses)) if losses else float('nan')
        logs.append(dict(strategy=strategy,fold=fold,epoch=epoch+1,masked_BCE=loss_epoch))
        pd.DataFrame(logs).to_csv(OUT/f'loss_{strategy}_fold{fold}.csv',index=False)
        print(f'{strategy} fold={fold} epoch={epoch+1}/{EPOCHS}: loss={loss_epoch:.4f} ({time.time()-start:.1f}s)',flush=True)
    torch.save(net.state_dict(),OUT/f'model_{strategy}_fold{fold}.pt')
    return net



## 06 — Evaluate complete held-out circle coverage

Reads native 512px validation windows, masks out everything outside the **specific** circle and general ROI, and counts valid RGB pixels only. Area fraction and Brier use **all** eligible pixels; ROC-AUC/AP use a reproducible diagnostic subsample per tile. There is no calibration fitted on held-out circles.


In [6]:
def score_full_circles(net,strategy,fold,site_filter=None):
    val=splits[(splits.cv_fold==fold)&(splits.split=='val')&(splits.kind=='exhaustive')]
    if site_filter is not None:
        val=val[val.site==site_filter]
    assert len(val)>0
    results={}
    net.eval()
    with torch.no_grad():
        for j,rec in enumerate(val.to_dict('records')):
            name=rec['mosaic'];cid=int(rec['circle_id'])
            window=Window(int(rec['col']),int(rec['row']),int(rec['width']),int(rec['height']))
            y,w,full,unc=make_masks(name,window,'val',include_partial=False)
            geom=circles[circles.original_row==cid].to_crs(geo[name]['crs'])
            transform=rasterio.windows.transform(window,geo[name]['transform'])
            bb=box(*rasterio.windows.bounds(window,geo[name]['transform']))
            clip=[g.intersection(bb) for g in geom.geometry if g.intersects(bb)]
            cmask=rasterize([(g,1) for g in clip if not g.is_empty],
                            out_shape=y.shape,transform=transform,fill=0,dtype='uint8').astype(bool) if clip else np.zeros_like(y,bool)
            with rasterio.open(geo[name]['path']) as ds:
                raw=ds.read([1,2,3],window=window,masked=True)
            valid=~np.any(np.ma.getmaskarray(raw),axis=0)
            raw=raw.filled(0).astype('float32')
            valid &= np.any(raw!=0,axis=0)&np.any(raw!=255,axis=0)
            eligible=(w>0)&cmask&valid
            if not eligible.any():continue
            x=np.clip(raw/255.,0,1)[None]
            pr=torch.sigmoid(net(torch.from_numpy(x).to(device)))[0].cpu().numpy()
            yy=y[eligible].astype('uint8');pp=pr[eligible]
            if cid not in results:results[cid]=dict(site=geo[name]['site'],n=0,sum_truth=0.,sum_prob=0.,
                    sum_sq=0.,tp=0,fp=0,fn=0,ys=[],ps=[],conflicts=0)
            v=results[cid];v['n']+=len(yy);v['sum_truth']+=float(yy.sum());v['sum_prob']+=float(pp.sum())
            v['sum_sq']+=float(np.square(yy-pp).sum())
            binary=pp>=.5
            v['tp']+=int(np.sum(binary&(yy==1)));v['fp']+=int(np.sum(binary&(yy==0)))
            v['fn']+=int(np.sum((~binary)&(yy==1)))
            v['conflicts']+=int(unc[eligible].sum())
            rng=np.random.default_rng(SEED+cid*1009+j)
            take=rng.choice(len(yy),min(len(yy),200),replace=False)
            v['ys'].extend(yy[take].tolist());v['ps'].extend(pp[take].tolist())
            if (j+1)%100==0:print('  evaluation windows:',j+1,'/',len(val),flush=True)
    scored=[]
    for cid,v in sorted(results.items()):
        yy=np.array(v['ys']);pp=np.array(v['ps']);nn=v['n']
        scored.append(dict(strategy=strategy,fold=fold,site=v['site'],circle_id=cid,
            reference_pixels=nn,observed_cover_pct=100*v['sum_truth']/nn,
            probability_cover_pct=100*v['sum_prob']/nn,
            bias_pp=100*(v['sum_prob']-v['sum_truth'])/nn,
            brier=v['sum_sq']/nn,f1_05=2*v['tp']/max(1,2*v['tp']+v['fp']+v['fn']),
            conflict_pct=100*v['conflicts']/nn,
            auc_sample=roc_auc_score(yy,pp) if len(set(yy))>1 else np.nan,
            ap_sample=average_precision_score(yy,pp) if len(set(yy))>1 else np.nan))
    scores=pd.DataFrame(scored)
    assert len(scores),f'No scores for fold {fold}'
    return scores




## 07 — Run/resume pooled experiments

Each model checkpoint and its exhaustive full-circle score are saved on Drive. The evaluation iterates over ALL exhaustive windows; no validation-window subsampling. The appendices below add site adaptation and LOSO.


In [7]:
for fold in FOLDS:
    needed=[s for s in STRATEGIES if not (OUT/f'scores_{s}_fold{fold}.csv').exists()]
    if not needed:
        print('Fold',fold,'already completed',flush=True);continue
    cache,_=prepare_fold(fold)
    for strategy in needed:
        checkpoint=OUT/f'model_{strategy}_fold{fold}.pt'
        score_file=OUT/f'scores_{strategy}_fold{fold}.csv'
        print('\n=== RUN',strategy,'FOLD',fold,'===',flush=True)
        if checkpoint.exists():
            net=TinyUNet().to(device)
            net.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True))
            print('Loaded existing model checkpoint',flush=True)
        else:net=train_strategy(strategy,fold,cache)
        scores=score_full_circles(net,strategy,fold)
        scores.to_csv(score_file,index=False)
        print('SAVED',score_file,'circle n=',len(scores),
              'MAE pp=',round(scores.bias_pp.abs().mean(),2),flush=True)
        del net
        torch.cuda.empty_cache()
    del cache
print('Completed all available overnight jobs.')



Fold 0 train chips 668 complete-circle validation windows 285
Train per site: {'MOS': 173, 'Yoko': 171, 'Baego': 166, 'Yangambi': 158}

=== RUN exhaustive FOLD 0 ===
exhaustive fold=0 epoch=1/16: loss=0.7701 (5.2s)
exhaustive fold=0 epoch=2/16: loss=0.6670 (2.6s)
exhaustive fold=0 epoch=3/16: loss=0.6233 (2.6s)
exhaustive fold=0 epoch=4/16: loss=0.5925 (2.6s)
exhaustive fold=0 epoch=5/16: loss=0.5581 (2.6s)
exhaustive fold=0 epoch=6/16: loss=0.5347 (2.6s)
exhaustive fold=0 epoch=7/16: loss=0.5146 (2.6s)
exhaustive fold=0 epoch=8/16: loss=0.4915 (2.6s)
exhaustive fold=0 epoch=9/16: loss=0.4933 (2.6s)
exhaustive fold=0 epoch=10/16: loss=0.4698 (2.6s)
exhaustive fold=0 epoch=11/16: loss=0.4618 (2.6s)
exhaustive fold=0 epoch=12/16: loss=0.4469 (2.6s)
exhaustive fold=0 epoch=13/16: loss=0.4527 (2.6s)
exhaustive fold=0 epoch=14/16: loss=0.4418 (2.6s)
exhaustive fold=0 epoch=15/16: loss=0.4289 (2.6s)
exhaustive fold=0 epoch=16/16: loss=0.4297 (2.6s)
  evaluation windows: 100 / 285
  evaluatio

## 08 — Compare results and check completion


In [8]:
available=[]
for fold in FOLDS:
    for strategy in STRATEGIES:
        file=OUT/f'scores_{strategy}_fold{fold}.csv'
        if file.exists():available.append(pd.read_csv(file))
assert available,'No completed run yet'
scores=pd.concat(available,ignore_index=True)
complete=scores.groupby('strategy').agg(folds=('fold','nunique'),circles=('circle_id','nunique'))
complete['complete']=complete.folds.eq(3)&complete.circles.eq(22)
display(complete)
by_site=scores.groupby(['strategy','site']).agg(circles=('circle_id','nunique'),
    bias_pp=('bias_pp','mean'),cover_mae_pp=('bias_pp',lambda s:float(s.abs().mean())),
    brier=('brier','mean'),auc=('auc_sample','mean'),ap=('ap_sample','mean'),
    f1=('f1_05','mean')).reset_index()
display(by_site.round(3))
overall=by_site.groupby('strategy').agg(macro_cover_mae_pp=('cover_mae_pp','mean'),
    macro_brier=('brier','mean'),macro_auc=('auc','mean'),macro_ap=('ap','mean')).sort_values('macro_cover_mae_pp')
display(overall.round(3))
scores.to_csv(OUT/'02B_circle_scores.csv',index=False)
by_site.to_csv(OUT/'02B_by_site.csv',index=False)
overall.to_csv(OUT/'02B_strategy_comparison.csv')
if not complete.complete.all():print('INCOMPLETE: rerun Block 07 to resume')
else:print('ALL 12 RUNS FINISHED')



,folds,circles,complete
strategy,,,
combined,3,22,True
combined_balanced,3,22,True
combined_light,3,22,True
exhaustive,3,22,True


,strategy,site,circles,bias_pp,cover_mae_pp,brier,auc,ap,f1
0,combined,Baego,3,-18.314,21.654,0.270,0.801,0.785,0.460
1,combined,MOS,7,-16.923,30.922,0.278,0.797,0.741,0.415
2,combined,Yangambi,7,1.953,9.633,0.118,0.718,0.288,0.173
3,combined,Yoko,5,-4.510,8.266,0.131,0.807,0.569,0.345
4,combined_balanced,Baego,3,-4.514,27.973,0.269,0.813,0.783,0.503
5,combined_balanced,MOS,7,-8.123,35.062,0.321,0.783,0.712,0.408
6,combined_balanced,Yangambi,7,6.843,13.272,0.135,0.718,0.300,0.241
7,combined_balanced,Yoko,5,5.587,10.546,0.133,0.828,0.595,0.530
8,combined_light,Baego,3,-13.915,22.990,0.228,0.795,0.772,0.514
9,combined_light,MOS,7,-8.983,23.004,0.195,0.804,0.760,0.601


,macro_cover_mae_pp,macro_brier,macro_auc,macro_ap
strategy,,,,
combined,17.619,0.199,0.781,0.596
combined_light,17.943,0.176,0.782,0.593
exhaustive,18.681,0.189,0.761,0.550
combined_balanced,21.713,0.215,0.786,0.598


ALL 12 RUNS FINISHED


## 09 — Site-specific adaptation: third production-mapping strategy

For each of three spatial folds and four sites, start from the corresponding **pooled combined** model, then fine-tune for a few epochs using only **training-fold chips from that site**. Score on the SAME site’s held-out exhaustive circles; never adapt using its held-out circles. Compare pooled vs adapted **paired by circle**. This tests the likely best final mapping approach without conflating it with transfer to unseen forests.


In [9]:
ADAPT_EPOCHS=6
ADAPT_LR=1e-4
def finetune_site(fold,site,items):
    base=OUT/f'model_combined_fold{fold}.pt'
    assert base.exists(),f'Run pooled combined fold {fold} first'
    rows=[q for q in items if q['site']==site and q['w'].sum()>200]
    assert len(rows)>=5,('Insufficient site adaptation data',site,fold,len(rows))
    torch.manual_seed(SEED+fold)
    model=TinyUNet().to(device)
    model.load_state_dict(torch.load(base,map_location=device,weights_only=True))
    opt=torch.optim.AdamW(model.parameters(),lr=ADAPT_LR,weight_decay=1e-4)
    loader=DataLoader(StrategyData(rows,'combined'),batch_size=BATCH,shuffle=True,num_workers=0)
    for epoch in range(ADAPT_EPOCHS):
        model.train();losses=[]
        for x,y,w in loader:
            x=x.to(device);y=y.to(device);w=w.to(device)
            if w.sum().item()==0:continue
            opt.zero_grad(set_to_none=True)
            loss=masked_loss(model(x),y,w)
            loss.backward();opt.step()
            losses.append(loss.item())
        print('adapt',site,'fold',fold,'epoch',epoch+1,'/',ADAPT_EPOCHS,
              'loss',round(float(np.mean(losses)),4),flush=True)
    return model

# Reuse complete-circle scoring while avoiding contamination by other sites.
def score_site_only(model,site,fold,strategy_label):
    return score_full_circles(model,strategy_label,fold,site_filter=site)

for fold in FOLDS:
    pending=[site for site in ['Baego','MOS','Yangambi','Yoko']
             if not (OUT/f'scores_adapt_{site}_fold{fold}.csv').exists()]
    if not pending:continue
    items,_=prepare_fold(fold)
    for site in pending:
        checkpoint=OUT/f'model_adapt_{site}_fold{fold}.pt'
        output=OUT/f'scores_adapt_{site}_fold{fold}.csv'
        if checkpoint.exists():
            model=TinyUNet().to(device)
            model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True))
        else:
            model=finetune_site(fold,site,items)
            torch.save(model.state_dict(),checkpoint)
        result=score_site_only(model,site,fold,'adapted')
        assert len(result)>0,('No evaluation circle',site,fold)
        result.to_csv(output,index=False)
        print('SAVED site-adapted',site,fold,'circles',len(result),flush=True)
        del model
        torch.cuda.empty_cache()
    del items




Fold 0 train chips 668 complete-circle validation windows 285
Train per site: {'MOS': 173, 'Yoko': 171, 'Baego': 166, 'Yangambi': 158}
adapt Baego fold 0 epoch 1 / 6 loss 0.3632
adapt Baego fold 0 epoch 2 / 6 loss 0.3718
adapt Baego fold 0 epoch 3 / 6 loss 0.3387
adapt Baego fold 0 epoch 4 / 6 loss 0.3568
adapt Baego fold 0 epoch 5 / 6 loss 0.3562
adapt Baego fold 0 epoch 6 / 6 loss 0.3229
SAVED site-adapted Baego 0 circles 1
adapt MOS fold 0 epoch 1 / 6 loss 0.3794
adapt MOS fold 0 epoch 2 / 6 loss 0.327
adapt MOS fold 0 epoch 3 / 6 loss 0.3608
adapt MOS fold 0 epoch 4 / 6 loss 0.3316
adapt MOS fold 0 epoch 5 / 6 loss 0.2966
adapt MOS fold 0 epoch 6 / 6 loss 0.3135
SAVED site-adapted MOS 0 circles 3
adapt Yangambi fold 0 epoch 1 / 6 loss 0.4611
adapt Yangambi fold 0 epoch 2 / 6 loss 0.4427
adapt Yangambi fold 0 epoch 3 / 6 loss 0.4355
adapt Yangambi fold 0 epoch 4 / 6 loss 0.4273
adapt Yangambi fold 0 epoch 5 / 6 loss 0.4196
adapt Yangambi fold 0 epoch 6 / 6 loss 0.4235
  evaluation w

## 10 — All-site LOSO transfer test (four fully excluded forests)

For each target forest, train on the **other three forest groups**, using their exhaustive and partial labels. No annotation from the target forest enters training, early stopping or probability calibration. The three MOS mosaics are always held out together. Test on **all exhaustive circles** of the excluded forest. LOSO is an informative *secondary* metric, not the final map-selection criterion.


In [10]:
# Build LOSO reference masks using the same original geometries and contradictory-pixel rule.
# Explicitly prevent ANY target-site annotation from entering LOSO training.
CANDIDATES=pd.read_csv(REF/'01_candidate_windows.csv')
LOSO_SITES=['Baego','MOS','Yangambi','Yoko']
LOSO_EPOCHS=16

def loso_training_chip(record,held_site):
    name=record['mosaic'];d=geo[name]
    if d['site']==held_site:return None
    if record['height']<CROP or record['width']<CROP:return None
    rr=int(record['row'])+(int(record['height'])-CROP)//2
    cc=int(record['col'])+(int(record['width'])-CROP)//2
    window=Window(cc,rr,CROP,CROP)
    shape=(CROP,CROP)
    bb=box(*rasterio.windows.bounds(window,d['transform']))
    transform=rasterio.windows.transform(window,d['transform'])
    def burn(gs):
        shapes=[g.intersection(bb) for g in gs if g is not None and g.intersects(bb)]
        if not shapes:return np.zeros(shape,bool)
        return rasterize([(g,1) for g in shapes if not g.is_empty],out_shape=shape,
                         transform=transform,fill=0,dtype='uint8').astype(bool)
    inside=burn(roi.to_crs(d['crs']).geometry)
    # All of the training sites' exhaustive circles, including all three MOS mosaics.
    owned=d['circles'];owned=owned[owned.owner==name]
    area=burn(owned.geometry)&inside
    pos=burn(d['positive'].geometry)&area
    t=d['trees'];tp=burn(t.loc[t.class01==1].geometry);tn=burn(t.loc[t.class01==0].geometry)
    contradiction=(pos&tn)|(area&~pos&tp)|(~area&tp&tn)
    label=(pos|(~area&tp)).astype('float32')
    supervision=(area|(~area&(tp|tn)))&inside&~contradiction
    with rasterio.open(d['path']) as ds:
        raw=ds.read([1,2,3],window=window,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    rgb=raw.filled(0).astype('float32')
    valid &= np.any(rgb!=0,axis=0)&np.any(rgb!=255,axis=0)
    supervision &= valid
    if supervision.sum()<300:return None
    return dict(rgb=np.clip(rgb/255.,0,1).transpose(1,2,0).astype('float32'),
                y=label,w=supervision.astype('float32'),full=area,unc=contradiction,
                site=d['site'],kind=record['kind'],name=name,circle_id=record['circle_id'])

def loso_score_all_circles(net,held_site):
    global FOLD
    # score_full_circles uses the standard held-out fold selector.
    # Assemble the four fold-specific pieces of validation manifest, then score
    # held-out site circles on their own actual folds; all are absent from training.
    outputs=[]
    for fold in FOLDS:
        FOLD=fold
        batch=score_full_circles(net,'LOSO_'+held_site,fold,site_filter=held_site)
        outputs.append(batch)
    combined=pd.concat(outputs,ignore_index=True)
    assert combined.circle_id.nunique()==int((links.site==held_site).sum()),(
        'LOSO should evaluate every exhaustive circle once',held_site)
    return combined

for held_site in LOSO_SITES:
    target=OUT/f'scores_LOSO_{held_site}.csv'
    checkpoint=OUT/f'model_LOSO_{held_site}.pt'
    if target.exists():
        print('LOSO',held_site,'already scored — skip',flush=True)
        continue
    if checkpoint.exists():
        net=TinyUNet().to(device)
        net.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True))
    else:
        candidates=CANDIDATES[CANDIDATES.site!=held_site].copy()
        pieces=[]
        for (site,kind),g in candidates.groupby(['site','kind']):
            subset=g.sample(n=min(len(g),MAX_PER_SITE_KIND*2),random_state=SEED)
            pieces+=subset.to_dict('records')
        cache=[]
        for i,rec in enumerate(pieces):
            q=loso_training_chip(rec,held_site)
            if q is not None:cache.append(q)
            if i and i%150==0:print('LOSO',held_site,'loaded',i,'/',len(pieces),flush=True)
        assert len(cache)>=20,(held_site,len(cache))
        print('LOSO',held_site,'training chips',len(cache),'sites',
              pd.Series([q['site'] for q in cache]).value_counts().to_dict(),flush=True)
        old_epochs=EPOCHS
        EPOCHS=LOSO_EPOCHS
        net=train_strategy('combined_balanced',-1,cache)
        EPOCHS=old_epochs
        torch.save(net.state_dict(),checkpoint)
        del cache
    result=loso_score_all_circles(net,held_site)
    result.to_csv(target,index=False)
    print('LOSO DONE',held_site,'circles',len(result),
          'fraction MAE',round(result.bias_pp.abs().mean(),2),flush=True)
    del net
    torch.cuda.empty_cache()




LOSO Baego loaded 150 / 952
LOSO Baego loaded 300 / 952
LOSO Baego loaded 450 / 952
LOSO Baego loaded 600 / 952
LOSO Baego loaded 750 / 952
LOSO Baego loaded 900 / 952
LOSO Baego training chips 886 sites {'Yoko': 340, 'Yangambi': 331, 'MOS': 215}
combined_balanced fold=-1 epoch=1/16: loss=0.5863 (13.6s)
combined_balanced fold=-1 epoch=2/16: loss=0.4605 (13.8s)
combined_balanced fold=-1 epoch=3/16: loss=0.4355 (13.9s)
combined_balanced fold=-1 epoch=4/16: loss=0.4551 (14.0s)
combined_balanced fold=-1 epoch=5/16: loss=0.4291 (13.9s)
combined_balanced fold=-1 epoch=6/16: loss=0.4422 (13.7s)
combined_balanced fold=-1 epoch=7/16: loss=0.4121 (13.7s)
combined_balanced fold=-1 epoch=8/16: loss=0.4219 (13.7s)
combined_balanced fold=-1 epoch=9/16: loss=0.4186 (13.8s)
combined_balanced fold=-1 epoch=10/16: loss=0.4155 (13.9s)
combined_balanced fold=-1 epoch=11/16: loss=0.3937 (13.8s)
combined_balanced fold=-1 epoch=12/16: loss=0.3863 (13.8s)
combined_balanced fold=-1 epoch=13/16: loss=0.4130 (13

## 11 — Integrated strategy comparison: production mapping versus cross-site transfer

Use the **within-site pooled and site-adapted CV** results to choose a full-mosaic mapping strategy. Report LOSO separately: LOSO evaluates a more difficult and different objective. No site-specific calibration may use a circle's held-out reference.


In [11]:
from pathlib import Path
pooled=[]
adapt=[]
loso=[]
for fold in FOLDS:
    for s in STRATEGIES:
        f=OUT/f'scores_{s}_fold{fold}.csv'
        if f.exists():pooled.append(pd.read_csv(f))
    for site in LOSO_SITES:
        f=OUT/f'scores_adapt_{site}_fold{fold}.csv'
        if f.exists():adapt.append(pd.read_csv(f))
for site in LOSO_SITES:
    f=OUT/f'scores_LOSO_{site}.csv'
    if f.exists():loso.append(pd.read_csv(f))
tables=[]
for experiment,frames in [('within_site_pooled',pooled),('within_site_adapted',adapt),('LOSO',loso)]:
    if not frames:continue
    d=pd.concat(frames,ignore_index=True)
    group=d.groupby('site').agg(circles=('circle_id','nunique'),
      fraction_MAE_pp=('bias_pp',lambda x:float(np.abs(x).mean())),
      fraction_bias_pp=('bias_pp','mean'),brier=('brier','mean'),
      AUC=('auc_sample','mean'),AP=('ap_sample','mean')).reset_index()
    if experiment=='within_site_pooled':
        group=d.groupby(['strategy','site']).agg(circles=('circle_id','nunique'),
          fraction_MAE_pp=('bias_pp',lambda x:float(np.abs(x).mean())),
          fraction_bias_pp=('bias_pp','mean'),brier=('brier','mean'),
          AUC=('auc_sample','mean'),AP=('ap_sample','mean')).reset_index()
    group.insert(0,'experiment',experiment)
    tables.append(group)
comparison=pd.concat(tables,ignore_index=True)
display(comparison.round(3))
comparison.to_csv(OUT/'02B_extended_strategy_comparison.csv',index=False)
print('Use within-site results for full UAV map selection; LOSO is transferability only.')



,experiment,strategy,site,circles,fraction_MAE_pp,fraction_bias_pp,brier,AUC,AP
0,within_site_pooled,combined,Baego,3,21.654,-18.314,0.270,0.801,0.785
1,within_site_pooled,combined,MOS,7,30.922,-16.923,0.278,0.797,0.741
2,within_site_pooled,combined,Yangambi,7,9.633,1.953,0.118,0.718,0.288
3,within_site_pooled,combined,Yoko,5,8.266,-4.510,0.131,0.807,0.569
4,within_site_pooled,combined_balanced,Baego,3,27.973,-4.514,0.269,0.813,0.783
5,within_site_pooled,combined_balanced,MOS,7,35.062,-8.123,0.321,0.783,0.712
6,within_site_pooled,combined_balanced,Yangambi,7,13.272,6.843,0.135,0.718,0.300
7,within_site_pooled,combined_balanced,Yoko,5,10.546,5.587,0.133,0.828,0.595
8,within_site_pooled,combined_light,Baego,3,22.990,-13.915,0.228,0.795,0.772
9,within_site_pooled,combined_light,MOS,7,23.004,-8.983,0.195,0.804,0.760


Use within-site results for full UAV map selection; LOSO is transferability only.
